# Sultai mana-base explorer

Run all cells once. Then edit **Shock penalty** and **Town penalty** in the parameter cell and rerun the analysis cell below it. The expensive draw calculations stay cached in the tables.

Each penalty is charged per matching land in the base; Cavern has no composition penalty. The starting values are illustrative, not calibrated life costs. Modes have uniform weights within each physical spell profile.

Basic and nonbasic bounds are editable below. Exhaustive builds use the batch backend without grid-size cutoffs. The candidate count and capacity-grid size are printed before building. Town and Cavern are separate unrestricted rainbow sources; Cavern has no penalty.

The report scores the full pool once, selects only the top ten rows, and reuses the score snapshot for casting-cost and penalty crossings.


In [29]:
from pathlib import Path
import sys
from time import perf_counter
from math import prod
from html import escape

# Works when launched from the repository or one of its subdirectories.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_opportunities.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from IPython.display import HTML, display
from tqdm.auto import tqdm
from mana_opportunities import CastingMode, ManaSource, Spell, SpellCost, VERGE_RULES, spell_components
from mana_search import ManaSpace, build_component_table


def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def source_name(source):
    return f'{source.kind}:{source.colors}'


def source_list(entries):
    return ', '.join(f'{n} {source_name(source)}' for source, n in entries) or 'none'


def number(value):
    return 'unbounded' if value is None else f'{float(value):.8g}'


def sensitivity_row(result):
    interval = result.optimal_interval
    below, above = result.lower, result.upper
    return (
        result.parameter, number(result.current_value), number(interval.lower),
        source_list(below.swap.incoming) if below else 'none',
        source_list(below.swap.out) if below else 'none',
        number(interval.upper),
        source_list(above.swap.incoming) if above else 'none',
        source_list(above.swap.out) if above else 'none',
    )


## Spell profiles and starting mana base

Edit these definitions here, then rerun this cell, build the search tables, and run the analysis. Counts in `MANA_BASE` describe the starting composition; the build cell allows 0?2 of each basic and caps shocks, Verges, Town, and Cavern at four each. The total land count is inferred from this composition. Mode weights default to uniform within each spell profile.

The shocks explicitly carry both basic land types. Verges, Town, and Cavern carry none. Verge secondary colors unlock when an enabling land type appears anywhere among lands seen, even if that land would not fit within the turn's land drops. This deliberately optimistic approximation avoids sequencing and battlefield search. Verges have no additional composition penalty.


In [30]:
SPELLS = (
    SpellCost("2UB", 4),  # Superior Spider-Man
    Spell(4, tuple(CastingMode(c) for c in ("UU", "BB", "4UU", "4BB", "2UUBB"))),  # Deceit
    Spell(4, tuple(CastingMode(c) for c in ("WW", "BB", "4WW", "4BB", "2WWBB"))),  # Emptiness
    Spell(4, tuple(CastingMode(c) for c in ("UU", "GG", "3UU", "3GG", "1UUGG"))),  # Wistfulness
    SpellCost("2G", 4),   # Formidable Speaker
    SpellCost("1B", 3),   # Bitter Triumph
    SpellCost("B", 4),    # Moonshadow
    SpellCost("7UU", 4),  # Sunderflock
    SpellCost("BGU", 3),  # Awaken the Honored Dead
)

MANA_BASE = {
    # ManaSource("U", "basic"): 0,
    ManaSource("B", "basic"): 2,
    # ManaSource("G", "basic"): 0,
    ManaSource("UB", "shock", types="UB"): 2,
    ManaSource("UG", "shock", types="UG"): 4,
    ManaSource("BG", "shock", types="BG"): 4,
    ManaSource("WUBRG", "town"): 4,
    ManaSource("WUBRG", "cavern"): 4,
    ManaSource("UB", "verge"): 2,
    ManaSource("BG", "verge"): 1,
    ManaSource("UG", "verge"): 3,
}


## Build search tables

Run this cell again only if you change the spell profile, bounds, or evaluation settings. The Sultai input uses an illustrative mana base with basics, typed shocks, separate Town/Cavern rainbow sources, and conditional Verges. Cavern restrictions are excluded.

Batch creation shares probability calculations across every base. Component bars report the expensive transforms; base bars report row assembly. The scalar backend remains available with `backend="scalar"` for exhaustive reference checks.


In [31]:

land_count = sum(MANA_BASE.values())
space = ManaSpace(
    {source: (0, 3 if source.kind == "basic" else 4) for source in MANA_BASE},
    land_count=land_count,
)
print(f'Exhaustive candidates: {space.candidate_count()}')
grid_cells = prod(hi + 1 for _, hi in space.bounds.values())
print(f'Capacity grid: {grid_cells:,} cells')
started = perf_counter()
table = build_component_table(
    space, SPELLS, baseline=MANA_BASE,
    progress=lambda candidates: tqdm(candidates, total=space.candidate_count(), desc='Base rows', unit='base'),
    component_progress=lambda costs: tqdm(costs, total=len(spell_components(SPELLS)[1]),
                                          desc='Components', unit='cost'),
)
print(f'{len(table.rows)} bases built in {perf_counter() - started:.2f}s')


Exhaustive candidates: 19071
Capacity grid: 1,562,500 cells


Components:   0%|          | 0/18 [00:00<?, ?cost/s]

Base rows:   0%|          | 0/19071 [00:00<?, ?base/s]

19071 bases built in 6.71s


## Parameters

Edit these two values, run this cell, then run the analysis cell. Zero disables that penalty. Changing these values does not require rebuilding the tables.


In [32]:
shock_penalty = 0.00687
town_penalty = 0.01  # Independent control; initially equal to shock_penalty.


## Analysis

The optimum appears first, followed by the ten highest-scoring bases. Candidate IN/OUT changes are relative to the original starting base. Cost-weight sensitivities follow, then penalty sensitivities.

Each sensitivity varies one absolute nonnegative parameter with all others fixed. The optimum stays optimal between the lower and upper boundaries (including ties). IN/OUT below or above a boundary describes the immediate successor relative to this optimum. A zero lower boundary has no transition into negative values. Exact fractions remain available in `analysis`.


In [33]:
penalties = dict(shock_penalty=shock_penalty, town_penalty=town_penalty)
started = perf_counter()
scored = table.score(**penalties)
ranked = scored.rank(limit=10)
optimum = ranked[0]
winner = dict(zip(space.sources, optimum.counts))

display(HTML('<h3>Optimum solution</h3>'))
print(f'{len(table.rows):,} candidates; shock penalty {shock_penalty}; town penalty {town_penalty}')
show_table(
    ['Raw opportunities', 'Penalty', 'Score', 'Gain from start'],
    [(f'{float(optimum.raw_score):.6f}', f'{float(optimum.penalty):.6f}',
      f'{float(optimum.score):.6f}', f'{float(optimum.delta):+.6f}')],
)
show_table(['Source', 'Count'],
           [(source_name(s), count) for s, count in winner.items() if count] + [('Total', sum(winner.values()))])

display(HTML('<h3>Top 10 candidates</h3>'))
show_table(
    ['Rank', 'Raw opportunities', 'Penalty', 'Score', 'Gain from start', 'IN from start', 'OUT from start'],
    [(i, f'{float(r.raw_score):.6f}', f'{float(r.penalty):.6f}', f'{float(r.score):.6f}',
      f'{float(r.delta):+.6f}', source_list(r.swap.incoming), source_list(r.swap.out))
     for i, r in enumerate(ranked, 1)],
)

display(HTML('<h3>Casting-cost weight sensitivity</h3>'))
cost_sensitivities = {cost: scored.crossings(cost, baseline=winner) for cost in table.costs}
sensitivity_headers = ['Parameter', 'Current', 'Lower boundary', 'IN below', 'OUT below',
                       'Upper boundary', 'IN above', 'OUT above']
show_table(sensitivity_headers, [sensitivity_row(result) for result in cost_sensitivities.values()])

display(HTML('<h3>Penalty sensitivity</h3>'))
penalty_sensitivities = {parameter: scored.crossings(parameter, baseline=winner) for parameter in penalties}
show_table(sensitivity_headers, [sensitivity_row(result) for result in penalty_sensitivities.values()])
analysis = dict(scored=scored, optimum=optimum, ranked=ranked,
                cost_sensitivities=cost_sensitivities, penalty_sensitivities=penalty_sensitivities)
print(f'Report computed in {perf_counter() - started:.2f}s')


19,071 candidates; shock penalty 0.00687; town penalty 0.01


Raw opportunities,Penalty,Score,Gain from start
3.252287,0.108700,3.143587,+0.009834


Source,Count
basic:B,2
cavern:WUBRG,4
shock:BG,3
shock:UB,4
shock:UG,3
town:WUBRG,4
verge:BG,2
verge:UB,2
verge:UG,2
Total,26


Rank,Raw opportunities,Penalty,Score,Gain from start,IN from start,OUT from start
1,3.252287,0.108700,3.143587,+0.009834,"2 shock:UB, 1 verge:BG","1 shock:BG, 1 shock:UG, 1 verge:UG"
2,3.252094,0.108700,3.143394,+0.009640,"2 shock:UB, 2 verge:BG","2 shock:BG, 2 verge:UG"
3,3.252070,0.108700,3.143370,+0.009617,"2 shock:UB, 1 verge:BG","2 shock:UG, 1 verge:UB"
4,3.258930,0.115570,3.143360,+0.009606,"2 shock:UB, 1 verge:BG, 1 verge:UB","1 basic:B, 1 shock:UG, 2 verge:UG"
5,3.258785,0.115570,3.143215,+0.009461,"2 shock:UB, 1 verge:BG","1 basic:B, 1 shock:UG, 1 verge:UG"
6,3.251873,0.108700,3.143173,+0.009419,2 shock:UB,2 shock:UG
7,3.258679,0.115570,3.143109,+0.009355,"2 shock:UB, 2 verge:BG, 1 verge:UB","1 basic:B, 1 shock:BG, 3 verge:UG"
8,3.251718,0.108700,3.143018,+0.009264,"2 shock:UB, 1 verge:BG","2 shock:UG, 1 verge:UG"
9,3.258512,0.115570,3.142942,+0.009189,"2 shock:UB, 1 verge:BG","1 shock:BG, 2 verge:UG"
10,3.258491,0.115570,3.142921,+0.009168,"1 shock:UB, 1 verge:BG, 2 verge:UB","1 basic:B, 3 verge:UG"


Parameter,Current,Lower boundary,IN below,OUT below,Upper boundary,IN above,OUT above
1B,1,0,none,none,1.4053742,"1 shock:BG, 1 verge:UB","1 basic:B, 1 verge:UG"
1UUGG,1,0,none,none,1.5259974,"1 shock:BG, 1 verge:UB","1 basic:B, 1 verge:UG"
2G,1,0.38389387,"1 shock:BG, 1 verge:UG","1 shock:UG, 1 verge:BG",1.2328787,"1 shock:BG, 1 verge:UG","1 shock:UG, 1 verge:UB"
2UB,1,0.28110257,"1 shock:BG, 1 verge:UG","1 shock:UG, 1 verge:UB",7.352616,"1 shock:UG, 1 verge:UB","1 basic:B, 1 verge:UG"
2UUBB,1,0,none,none,69.260123,"1 shock:UG, 1 verge:UB","1 basic:B, 1 verge:UG"
2WWBB,1,0,none,none,2.6212302,"1 shock:BG, 1 verge:UB","1 basic:B, 1 verge:UG"
3GG,1,0,none,none,2.0164007,"1 shock:BG, 1 verge:UG","1 shock:UG, 1 verge:UB"
3UU,1,0,none,none,9.3425059,"1 shock:UG, 1 verge:UB","1 basic:B, 1 verge:UG"
4BB,1,0,none,none,113.11677,"1 shock:BG, 1 verge:UB","1 basic:B, 1 verge:UG"
4UU,1,0,none,none,69.260123,"1 shock:UG, 1 verge:UB","1 basic:B, 1 verge:UG"


Parameter,Current,Lower boundary,IN below,OUT below,Upper boundary,IN above,OUT above
shock_penalty,0.00687,0.0066427325,"1 shock:BG, 1 verge:UB","1 basic:B, 1 verge:UG",0.0078791436,"1 shock:BG, 2 verge:UG","2 shock:UG, 1 verge:UB"
town_penalty,0.01,0,none,none,0.025201261,"1 shock:BG, 1 shock:UG, 1 verge:UB","1 basic:B, 1 town:WUBRG, 1 verge:UG"


Report computed in 0.23s
